#### Import Packages

In [ ]:
!pip install -U transformers tensorflow
#!pip uninstall -y transformers
#!pip install transformers[tf] tensorflow
!pip install torch transformers scikit-learn
import pandas
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import word_tokenize
!pip install pyspellchecker
from spellchecker import SpellChecker
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk import FreqDist
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.stem import PorterStemmer
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_curve,
    auc,
    RocCurveDisplay
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Bidirectional
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForSequenceClassification
from torch.optim import AdamW
import sklearn

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


#### Upload the dataset

In [ ]:
data = pandas.read_csv("/content/drive/MyDrive/chatgpt_style_reviews_dataset.csv")
data.head(5)

,date,title,review,rating,username,helpful_votes,review_length,platform,language,location,version,verified_purchase
0,########,Review title 1,"Not satisfied, many bugs and issues.",1,user1,80,6,Amazon,zh,Kenya,2.1.4,No
1,########,Review title 2,Amazing quality and user-friendly interface.,5,user2,180,5,Flipkart,zh,France,1.2.3,No
2,########,Review title 3,"Terrible experience, needs major improvements.",2,user3,154,5,Flipkart,pt,USA,1.2.3,No
3,########,Review title 4,Poor performance and not user-friendly.,1,user4,96,5,Amazon,es,Qatar,2.1.4,Yes
4,########,Review title 5,"Not satisfied, many bugs and issues.",2,user5,139,6,Website,ar,Kenya,2.1.4,No


In [ ]:
data.shape

(500, 12)

#### Required Columns - new data set

In [ ]:
new_data = data[['review','rating']]
new_data.head()

,review,rating
0,"Not satisfied, many bugs and issues.",1
1,Amazing quality and user-friendly interface.,5
2,"Terrible experience, needs major improvements.",2
3,Poor performance and not user-friendly.,1
4,"Not satisfied, many bugs and issues.",2


#### Remove Chat words

In [ ]:
chat_words_str = """
AFAIK=As Far As I Know
AFK=Away From Keyboard
ASAP=As Soon As Possible
ATK=At The Keyboard
ATM=At The Moment
A3=Anytime, Anywhere, Anyplace
BAK=Back At Keyboard
BBL=Be Back Later
BBS=Be Back Soon
BFN=Bye For Now
B4N=Bye For Now
BRB=Be Right Back
BRT=Be Right There
BTW=By The Way
B4=Before
B4N=Bye For Now
CU=See You
CUL8R=See You Later
CYA=See You
FAQ=Frequently Asked Questions
FC=Fingers Crossed
FWIW=For What It's Worth
FYI=For Your Information
GAL=Get A Life
GG=Good Game
GN=Good Night
GMTA=Great Minds Think Alike
GR8=Great!
G9=Genius
IC=I See
ICQ=I Seek you (also a chat program)
ILU=ILU: I Love You
IMHO=In My Honest/Humble Opinion
IMO=In My Opinion
IOW=In Other Words
IRL=In Real Life
KISS=Keep It Simple, Stupid
LDR=Long Distance Relationship
LMAO=Laugh My A.. Off
LMK=Let Me Know
LOL=Laughing Out Loud
LTNS=Long Time No See
L8R=Later
MTE=My Thoughts Exactly
M8=Mate
NRN=No Reply Necessary
OIC=Oh I See
PITA=Pain In The A..
PRT=Party
PRW=Parents Are Watching
ROFL=Rolling On The Floor Laughing
ROFLOL=Rolling On The Floor Laughing Out Loud
ROTFLMAO=Rolling On The Floor Laughing My A.. Off
SK8=Skate
STATS=Your sex and age
ASL=Age, Sex, Location
THX=Thank You
TTFN=Ta-Ta For Now!
TTYL=Talk To You Later
U=You
U2=You Too
U4E=Yours For Ever
WB=Welcome Back
WTF=What The F...
WTG=Way To Go!
WUF=Where Are You From?
W8=Wait...
7K=Sick:-D Laugher
"""

In [ ]:
chat_words_map_dict = {}
chat_words_list = []
for line in chat_words_str.split("\n"):
    if line != "":
        cw = line.split("=")[0]
        cw_expanded = line.split("=")[1]
        chat_words_list.append(cw)
        chat_words_map_dict[cw] = cw_expanded
chat_words_list = set(chat_words_list)

def chat_words_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words_list:
            new_text.append(chat_words_map_dict[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)
new_data['review']=new_data['review'].apply(chat_words_conversion)
new_data.head()

/tmp/ipython-input-827269242.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(chat_words_conversion)


,review,rating
0,"Not satisfied, many bugs and issues.",1
1,Amazing quality and user-friendly interface.,5
2,"Terrible experience, needs major improvements.",2
3,Poor performance and not user-friendly.,1
4,"Not satisfied, many bugs and issues.",2


#### Lower case

In [ ]:
def lower_case(text):
  return text.lower()
new_data['review']=new_data['review'].str.lower()
new_data.head()

/tmp/ipython-input-1730133729.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].str.lower()


,review,rating
0,"not satisfied, many bugs and issues.",1
1,amazing quality and user-friendly interface.,5
2,"terrible experience, needs major improvements.",2
3,poor performance and not user-friendly.,1
4,"not satisfied, many bugs and issues.",2


#### Removing white spaces

In [ ]:
def remove_whitespace(text):
  return " ".join(text.split())

new_data['review']=new_data['review'].apply(remove_whitespace)
new_data.head()

/tmp/ipython-input-222662379.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(remove_whitespace)


,review,rating
0,"not satisfied, many bugs and issues.",1
1,amazing quality and user-friendly interface.,5
2,"terrible experience, needs major improvements.",2
3,poor performance and not user-friendly.,1
4,"not satisfied, many bugs and issues.",2


#### Tokenization

In [ ]:
def tokenize(text):
  return word_tokenize(text)
new_data['review']=new_data['review'].apply(lambda X: word_tokenize(X))
new_data.head()

/tmp/ipython-input-2039697671.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(lambda X: word_tokenize(X))


,review,rating
0,"[not, satisfied, ,, many, bugs, and, issues, .]",1
1,"[amazing, quality, and, user-friendly, interfa...",5
2,"[terrible, experience, ,, needs, major, improv...",2
3,"[poor, performance, and, not, user-friendly, .]",1
4,"[not, satisfied, ,, many, bugs, and, issues, .]",2


#### Spelling correction

In [ ]:
def spell_check(text):
  try:
    result = []
    spell = SpellChecker()
    for word in text:
      correct_word = spell.correction(word)
      result.append(correct_word)
    # return " ".join(result)
    return result
  except:
    return""
new_data['review']=new_data['review'].apply(spell_check)
new_data.head()

/tmp/ipython-input-89934225.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(spell_check)


,review,rating
0,"[not, satisfied, ,, many, bugs, and, issues, .]",1
1,"[amazing, quality, and, None, interface, .]",5
2,"[terrible, experience, ,, needs, major, improv...",2
3,"[poor, performance, and, not, None, .]",1
4,"[not, satisfied, ,, many, bugs, and, issues, .]",2


#### Remove Stop words

In [ ]:
en_stopwords = set(stopwords.words('english'))
exceptional_words = {
    "not", "no", "nor", "never", "none", "nobody", "nothing",
    "hardly", "barely", "rarely",
    "very", "too", "so", "quite", "really", "extremely",
    "but", "however", "although", "though", "yet",
    "more", "less", "most", "least",
    "only", "just", "even"
}

en_stopwords = en_stopwords - exceptional_words

def remove_stopwords(text):
  result = []
  for token in text:
    # token= token.lower()
    if token not in en_stopwords:
      result.append(token)
  # return " ".join(result)
  return result

new_data['review']=new_data['review'].apply(remove_stopwords)
new_data.head()

/tmp/ipython-input-1134334137.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(remove_stopwords)


,review,rating
0,"[not, satisfied, ,, many, bugs, issues, .]",1
1,"[amazing, quality, None, interface, .]",5
2,"[terrible, experience, ,, needs, major, improv...",2
3,"[poor, performance, not, None, .]",1
4,"[not, satisfied, ,, many, bugs, issues, .]",2


#### Remove Empty words

In [ ]:
def remove_empty_words(text):
  if not text:
      return []
  return [w for w in text if isinstance(w, str) and w.strip()]

new_data['review'] = new_data['review'].apply(remove_empty_words)
new_data.head()

/tmp/ipython-input-4231054007.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review'] = new_data['review'].apply(remove_empty_words)


,review,rating
0,"[not, satisfied, ,, many, bugs, issues, .]",1
1,"[amazing, quality, interface, .]",5
2,"[terrible, experience, ,, needs, major, improv...",2
3,"[poor, performance, not, .]",1
4,"[not, satisfied, ,, many, bugs, issues, .]",2


#### Remove punctuations

In [ ]:
def remove_punct(text):
  tokenizer = RegexpTokenizer(r"\w+")
  lst=tokenizer.tokenize(' '.join(text))
  return lst

new_data['review']=new_data['review'].apply(remove_punct)
new_data.head()

/tmp/ipython-input-1784536471.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(remove_punct)


,review,rating
0,"[not, satisfied, many, bugs, issues]",1
1,"[amazing, quality, interface]",5
2,"[terrible, experience, needs, major, improveme...",2
3,"[poor, performance, not]",1
4,"[not, satisfied, many, bugs, issues]",2


#### Lemmatization

In [ ]:
def lemmatization(text):
  result=[]
  wordnet = WordNetLemmatizer()
  for token,tag in pos_tag(text):
    # print(token)
    # print(tag)
    pos=tag[0].lower()
    if pos not in ['a', 'r', 'n', 'v']:
      pos='n'
    result.append(wordnet.lemmatize(token,pos))
  return result
new_data['review']=new_data['review'].apply(lemmatization)
new_data.head()

/tmp/ipython-input-1823506958.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(lemmatization)


,review,rating
0,"[not, satisfy, many, bug, issue]",1
1,"[amazing, quality, interface]",5
2,"[terrible, experience, need, major, improvement]",2
3,"[poor, performance, not]",1
4,"[not, satisfy, many, bug, issue]",2


#### Remove Tags

In [ ]:
def remove_tag(text):
   text=' '.join(text)
   html_pattern = re.compile('<.*?>')
   return html_pattern.sub(r'', text)
new_data['review']=new_data['review'].apply(remove_tag)
new_data.head()

/tmp/ipython-input-1847718461.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['review']=new_data['review'].apply(remove_tag)


,review,rating
0,not satisfy many bug issue,1
1,amazing quality interface,5
2,terrible experience need major improvement,2
3,poor performance not,1
4,not satisfy many bug issue,2


#### Encoding rating

In [ ]:
def rating_to_sentiment(rating):
    if rating in [1, 2]:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    elif rating in [4, 5]:
        return "Positive"

new_data['rating'] = new_data['rating'].apply(rating_to_sentiment)
new_data.head()

/tmp/ipython-input-3898812438.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['rating'] = new_data['rating'].apply(rating_to_sentiment)


,review,rating
0,not satisfy many bug issue,Negative
1,amazing quality interface,Positive
2,terrible experience need major improvement,Negative
3,poor performance not,Negative
4,not satisfy many bug issue,Negative


In [ ]:
def encode_rating(rating):
  le = LabelEncoder()
  return le.fit_transform(rating)

new_data['rating'] = encode_rating(new_data['rating'])
new_data.head()

/tmp/ipython-input-1868199875.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['rating'] = encode_rating(new_data['rating'])


,review,rating
0,not satisfy many bug issue,0
1,amazing quality interface,2
2,terrible experience need major improvement,0
3,poor performance not,0
4,not satisfy many bug issue,0


#### Independent and Target

In [ ]:
X = new_data['review']
y = new_data['rating']

#### Train Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### ML - Logistic Regression

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ('logreg', LogisticRegression(multi_class='multinomial', max_iter=500, random_state=42))
])

param_grid = {
    'logreg__C': [0.001, 0.01, 0.1, 1, 10],
    'logreg__solver': ['lbfgs']
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

# Train model
grid.fit(X_train, y_train)

# Best model
best_logreg = grid.best_estimator_

# Predictions
y_pred = best_logreg.predict(X_test)
y_pred_prob = best_logreg.predict_proba(X_test)

# Metrics
accuracy_LR  = accuracy_score(y_test, y_pred)
precision_LR = precision_score(y_test, y_pred, average='macro')
recall_LR    = recall_score(y_test, y_pred, average='macro')
f1_LR        = f1_score(y_test, y_pred, average='macro')

print(f"Accuracy      : {accuracy_LR:.4f}")
print(f"Precision     : {precision_LR:.4f}")
print(f"Recall        : {recall_LR:.4f}")
print(f"F1 Score      : {f1_LR:.4f}")

print(classification_report(y_test, y_pred))

Accuracy      : 1.0000
Precision     : 1.0000
Recall        : 1.0000
F1 Score      : 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       1.00      1.00      1.00        25
           2       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


#### LR Pickle

In [ ]:
import pickle
filename = 'sentiment_analysis_LR.pkl'
pickle.dump(best_logreg, open(filename, 'wb'))

#### ML - Random Forest

In [ ]:
rf_cv = Pipeline(steps=[
    ("tfidf", TfidfVectorizer()),
    ('rf', RandomForestClassifier(random_state=42))
])

# Parameter grid (CV happens here)
param_grid = {
    'rf__n_estimators': [200, 300, 400],
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2]
}

# GridSearchCV (Cross-Validation)
rf_cv = GridSearchCV(
    rf_cv,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

# Train model
rf_cv.fit(X_train, y_train)

# Predictions
y_pred = rf_cv.predict(X_test)
y_pred_prob = rf_cv.predict_proba(X_test)

# Metrics
accuracy_RF  = accuracy_score(y_test, y_pred)
precision_RF = precision_score(y_test, y_pred, average='macro')
recall_RF    = recall_score(y_test, y_pred, average='macro')
f1_RF        = f1_score(y_test, y_pred, average='macro')

# Results
print(f"Accuracy      : {accuracy_RF:.4f}")
print(f"Precision     : {precision_RF:.4f}")
print(f"Recall        : {recall_RF:.4f}")
print(f"F1 Score      : {f1_RF:.4f}")
print(classification_report(y_test, y_pred))

Accuracy      : 1.0000
Precision     : 1.0000
Recall        : 1.0000
F1 Score      : 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       1.00      1.00      1.00        25
           2       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



#### ML - Naive Bayes

In [ ]:
nb_cv = Pipeline(steps=[
    ("tfidf", TfidfVectorizer()),
    ("nb", MultinomialNB())
])

# Parameter grid
param_grid = {
    "nb__alpha": [0.01, 0.1, 0.5, 1.0]
}

# GridSearchCV
nb_cv = GridSearchCV(
    nb_cv,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)


# Train model
nb_cv.fit(X_train, y_train)

# Predictions
y_pred = nb_cv.predict(X_test)
y_pred_prob = nb_cv.predict_proba(X_test)

# Metrics
accuracy_NB  = accuracy_score(y_test, y_pred)
precision_NB = precision_score(y_test, y_pred, average="macro")
recall_NB    = recall_score(y_test, y_pred, average="macro")
f1_NB        = f1_score(y_test, y_pred, average="macro")

# Results
print(f"Accuracy      : {accuracy_NB:.4f}")
print(f"Precision     : {precision_NB:.4f}")
print(f"Recall        : {recall_NB:.4f}")
print(f"F1 Score      : {f1_NB:.4f}")
print(classification_report(y_test, y_pred))

Accuracy      : 1.0000
Precision     : 1.0000
Recall        : 1.0000
F1 Score      : 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       1.00      1.00      1.00        25
           2       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



#### Pickle File

In [ ]:
import pickle
with open("nb_sentiment_model.pkl", "wb") as f:
    pickle.dump(nb_cv, f)

In [ ]:
import joblib

joblib.dump(nb_cv.best_estimator_, "nb_sentiment_analysis.pkl")
print(type(nb_cv))
print(type(nb_cv.best_estimator_))
print(sklearn.__version__)

<class 'sklearn.model_selection._search.GridSearchCV'>
<class 'imblearn.pipeline.Pipeline'>
1.6.1


#### DL - LSTMs

In [ ]:
# Tokenization and padding
MAX_WORDS = 20000
MAX_LEN = 300

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_test_pad  = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post")
# LSTM
num_classes = len(np.unique(y_train))

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=300, input_length=MAX_LEN, trainable = False),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.5),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=Adam(learning_rate=0.0005),
    metrics=["accuracy"]
)
# Handle imbalance
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
class_weight=class_weight_dict
# Train Model
history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    class_weight=class_weight_dict,
    verbose=1
)
# Prediction
y_pred_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

# Metrics
accuracy_LSTM  = accuracy_score(y_test, y_pred)
precision_LSTM = precision_score(y_test, y_pred, average="macro")
recall_LSTM    = recall_score(y_test, y_pred, average="macro")
f1_LSTM        = f1_score(y_test, y_pred, average="macro")

# Results
print(f"Accuracy      : {accuracy_LSTM:.4f}")
print(f"Precision     : {precision_LSTM:.4f}")
print(f"Recall        : {recall_LSTM:.4f}")
print(f"F1 Score      : {f1_LSTM:.4f}")
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 25s 3s/step - accuracy: 0.3582 - loss: 1.0969 - val_accuracy: 0.3250 - val_loss: 1.0952
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 15s 3s/step - accuracy: 0.4427 - loss: 1.0834 - val_accuracy: 0.3250 - val_loss: 1.0814
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 15s 3s/step - accuracy: 0.4695 - loss: 1.0796 - val_accuracy: 0.5000 - val_loss: 1.0739
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 22s 3s/step - accuracy: 0.5717 - loss: 1.0509 - val_accuracy: 0.4125 - val_loss: 1.0530
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.5512 - loss: 1.0346 - val_accuracy: 0.6000 - val_loss: 1.0300
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.6170 - loss: 1.0058 - val_accuracy: 0.6625 - val_loss: 1.0106
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 15s 3s/step - accuracy: 0.6141 - loss: 1.0102 - val_accuracy: 0.8250 - val_loss: 0.9740
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.6649 - loss: 0.9506 - val_accuracy: 0.7750 - val_loss: 0.8736
Epoch 9/

#### Transformer-based architectures

In [ ]:
# Device / GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#Tokenization
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

X_train_enc = tokenizer(
    list(X_train),
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

X_test_enc = tokenizer(
    list(X_test),
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

# Dataset class
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

y_train = y_train.values
y_test  = y_test.values

train_dataset = TextDataset(X_train_enc, y_train)
test_dataset  = TextDataset(X_test_enc, y_test)

# Data Loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Model
num_classes = len(np.unique(y_train))

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)

model.to(device)

# Optimizer + Class weight
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training
EPOCHS = 3

model.train()
for epoch in range(EPOCHS):
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")

# Prediction
model.eval()
y_pred = []
y_true = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.cpu().numpy())

# Metrics
accuracy_TR  = accuracy_score(y_true, y_pred)
precision_TR = precision_score(y_true, y_pred, average="macro")
recall_TR    = recall_score(y_true, y_pred, average="macro")
f1_TR        = f1_score(y_true, y_pred, average="macro")

print(f"Accuracy      : {accuracy_TR:.4f}")
print(f"Precision     : {precision_TR:.4f}")
print(f"Recall        : {recall_TR:.4f}")
print(f"F1 Score      : {f1_TR:.4f}")
print(classification_report(y_true, y_pred))

Using device: cpu


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 - Loss: 20.0447
Epoch 2/3 - Loss: 3.9797
Epoch 3/3 - Loss: 0.7355
Accuracy      : 1.0000
Precision     : 1.0000
Recall        : 1.0000
F1 Score      : 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       1.00      1.00      1.00        25
           2       1.00      1.00      1.00        39

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

